# 04 — Train: XGBoost regression

Fits one fixed direct 24-hour XGBoost model for the configured target station and evaluates it once on the test feature artifact.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview, model configuration, and test metrics

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports dependencies and fixes the notebook's configuration: one fixed native multi-output XGBoost configuration, the artifact path, preview row count, and feature/target column lists.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor

from src.config import TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
XGB_N_ESTIMATORS = 300
XGB_MAX_DEPTH = 6
XGB_LEARNING_RATE = 0.05
XGB_SUBSAMPLE = 0.8
XGB_COLSAMPLE_BYTREE = 0.8
XGB_OBJECTIVE = "reg:squarederror"
XGB_TREE_METHOD = "hist"
XGB_MULTI_STRATEGY = "one_output_per_tree"
XGB_RANDOM_STATE = 42
XGB_N_JOBS = -1
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())

## Shared evaluation cohort

The model is fit and scored only on rows with complete target vectors marked `target_valid` and complete predictor vectors. This excludes feature warm-up rows from both artifacts.

## Helper functions

Three helpers used by the load/fit/evaluate cells below: an eligibility check, metric tables, and a prediction preview.

In [ ]:
def eligible_rows(frame: pd.DataFrame, *, station_id: str, artifact_name: str) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(axis=1)
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [{
            "station_id": station_id,
            "scored_issue_times": len(actual),
            "scored_values": actual.size,
            "mae": mean_absolute_error(actual.to_numpy().ravel(), predictions.ravel()),
            "rmse": root_mean_squared_error(actual.to_numpy().ravel(), predictions.ravel()),
        }]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(actual[target], predictions[:, horizon - 1]),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon

In [ ]:
def prediction_preview(
    frame: pd.DataFrame, predictions: np.ndarray
) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load feature artifacts

Resolve the train/test parquet paths for the target station, failing fast if either is missing.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)

## Apply the eligibility cohort

Restrict both frames to the rows defined above, and stop early if a split has no usable rows.

In [ ]:
train_mask = eligible_rows(
    train_features, station_id=station_id, artifact_name="train"
)
test_mask = eligible_rows(
    test_features, station_id=station_id, artifact_name="test"
)
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

train_rows = train_features.loc[train_mask]
test_rows = test_features.loc[test_mask]

## Fit the fixed native multi-output XGBoost model

Fit one `XGBRegressor` across all 24 targets with XGBoost's native `one_output_per_tree` strategy, then predict once on the test cohort. Raw numeric feature values are passed directly to the trees: there is no scaling, imputation, tuning, validation split, early stopping, or second model.

In [ ]:
xgboost_model = XGBRegressor(
    n_estimators=XGB_N_ESTIMATORS,
    max_depth=XGB_MAX_DEPTH,
    learning_rate=XGB_LEARNING_RATE,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE_BYTREE,
    objective=XGB_OBJECTIVE,
    tree_method=XGB_TREE_METHOD,
    multi_strategy=XGB_MULTI_STRATEGY,
    random_state=XGB_RANDOM_STATE,
    n_jobs=XGB_N_JOBS,
)
xgboost_model.fit(train_rows[FEATURE_COLUMNS], train_rows[TARGET_COLUMNS])
test_predictions = xgboost_model.predict(test_rows[FEATURE_COLUMNS])

## Evaluate on the test cohort

Build aggregate and per-horizon metric tables, display the fixed model configuration, and preview five predictions against their actual targets.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
model_configuration = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "model": "XGBRegressor",
            "n_estimators": XGB_N_ESTIMATORS,
            "max_depth": XGB_MAX_DEPTH,
            "learning_rate": XGB_LEARNING_RATE,
            "subsample": XGB_SUBSAMPLE,
            "colsample_bytree": XGB_COLSAMPLE_BYTREE,
            "objective": XGB_OBJECTIVE,
            "tree_method": XGB_TREE_METHOD,
            "multi_strategy": XGB_MULTI_STRATEGY,
            "random_state": XGB_RANDOM_STATE,
            "n_jobs": XGB_N_JOBS,
        }
    ]
)
print(f"XGBoost test results for {station_id}")
display(model_configuration)
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))